# Modül 06: CNN — Evriimsel Sinir Ağları
## Deep Learning Path — Kapsamlı Jupyter Notebook
---
## 📋 İçindekiler
1. [Öğrenme Hedefleri](#1)
2. [Konvolüsyon Matematiği](#2)
3. [NumPy From Scratch](#3)
4. [Pooling Katmanları](#4)
5. [Klasik Mimariler](#5)
6. [ResNet Skip Connection](#6)
7. [Feature Map Görselleştirme](#7)
8. [Grad-CAM](#8)
9. [TF / PyTorch](#9)
10. [Alıştırmalar](#10)


## 1. Öğrenme Hedefleri <a id='1'></a>
- ✅ Konvolüsyon işlemini matematiksel olarak tanımlamak ve uygulamak
- ✅ Padding, stride ve dilation parametrelerini hesaplamak
- ✅ Çıkış boyutu formülünü doğru uygulamak
- ✅ Max ve Average Pooling farkını açıklamak
- ✅ LeNet'ten EfficientNet'e mimari evrimini özetlemek
- ✅ ResNet Skip Connection'ın neden çalıştığını açıklamak
- ✅ Grad-CAM ile model yorumlanabilirliği sağlamak


## 2. Konvolüsyon Matematiği <a id='2'></a>

Konvolüsyon işlemi:
$$(I * K)[i,j] = \sum_m \sum_n I[i+m,\, j+n] \cdot K[m,n]$$

Çıkış boyutu:
$$H_{out} = \left\lfloor\frac{H_{in} + 2P - K}{S}\right\rfloor + 1$$

| Parametre | Açıklama | Tipik Değer |
|-----------|----------|-------------|
| K (kernel) | Filtre boyutu | 3×3 veya 5×5 |
| P (padding) | Kenar dolgu | 0 (valid) veya 1 (same) |
| S (stride) | Kaydırma adımı | 1 (normal) veya 2 (küçültme) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

def conv2d(x, kernel, padding=0, stride=1):
    H,W = x.shape; kH,kW = kernel.shape
    if padding > 0:
        x = np.pad(x, padding, mode='constant')
    Ho = (H+2*padding-kH)//stride+1
    Wo = (W+2*padding-kW)//stride+1
    out = np.zeros((Ho,Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i,j] = np.sum(x[i*stride:i*stride+kH, j*stride:j*stride+kW]*kernel)
    return out

def max_pool(x, size=2, stride=2):
    H,W = x.shape
    Ho=(H-size)//stride+1; Wo=(W-size)//stride+1
    out=np.zeros((Ho,Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i,j]=np.max(x[i*stride:i*stride+size, j*stride:j*stride+size])
    return out

# Test görüntüsü
image = np.array([
    [0,0,0,0,0,0,0,0],
    [0,1,1,1,1,0,0,0],
    [0,1,0,0,1,0,0,0],
    [0,1,0,0,1,0,0,0],
    [0,1,1,1,1,0,0,0],
    [0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0],
],dtype=float)

edge_h = np.array([[-1,-1,-1],[0,0,0],[1,1,1]],dtype=float)
edge_v = np.array([[-1,0,1],[-1,0,1],[-1,0,1]],dtype=float)
blur   = np.ones((3,3))/9

# Boyut hesaplama
print("Boyut Hesaplama Tablosu:")
print(f"{'H_in':>6} {'K':>4} {'P':>4} {'S':>4} {'H_out':>7}")
print("-"*30)
for K,P,S in [(3,0,1),(3,1,1),(5,2,1),(3,0,2),(7,3,1)]:
    Ho=(8+2*P-K)//S+1
    print(f"{8:>6} {K:>4} {P:>4} {S:>4} {Ho:>7}")


In [ ]:
# Feature map görselleştirme
fmaps = {
    'Orijinal':        image,
    'Yatay Kenar':     conv2d(image, edge_h, padding=1),
    'Dikey Kenar':     conv2d(image, edge_v, padding=1),
    'Bulanık':         conv2d(image, blur,   padding=1),
    'Max Pool':        max_pool(conv2d(image, edge_v, padding=1)),
}
fig,axes=plt.subplots(1,5,figsize=(14,3))
for ax,(name,fm) in zip(axes,fmaps.items()):
    ax.imshow(fm,cmap='gray'); ax.set_title(name,fontsize=9,fontweight='bold'); ax.axis('off')
plt.suptitle('Konvolüsyon Feature Maps',fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Klasik Mimariler <a id='5'></a>

| Model | Yıl | Temel Yenilik | ImageNet Top-5 |
|-------|-----|---------------|----------------|
| AlexNet | 2012 | GPU, ReLU, Dropout | 15.3% |
| VGG-16 | 2014 | 3×3 filtreler, derinlik | 7.3% |
| GoogLeNet | 2014 | Inception modülü | 6.7% |
| ResNet-50 | 2015 | Skip connection | 3.57% |
| EfficientNet-B7 | 2019 | Compound scaling | 1.8% |


In [ ]:
# ResNet Skip Connection
def relu(x): return np.maximum(0,x)

def resnet_block(x, W1, W2):
    h   = relu(x @ W1)
    Fx  = h @ W2
    return relu(Fx + x)    # Skip connection!

np.random.seed(1)
dim=16; W1=np.random.randn(dim,dim)*.1; W2=np.random.randn(dim,dim)*.1
x0=np.random.randn(1,dim)
x_normal=x0.copy(); x_resnet=x0.copy()
for _ in range(20):
    x_normal = relu(relu(x_normal@W1)@W2)
    x_resnet = resnet_block(x_resnet,W1,W2)

print(f"20 blok sonrası sinyal normu:")
print(f"  Normal ağ : {np.linalg.norm(x_normal):.6f}")
print(f"  ResNet    : {np.linalg.norm(x_resnet):.6f}")
print("✓ ResNet skip connection sinyali canlı tutuyor!")


## 4. Grad-CAM <a id='8'></a>

Son konvolüsyon katmanının gradyanlarını kullanarak modelin hangi bölgeye odaklandığını görselleştir.

In [ ]:
# Grad-CAM simülasyonu
np.random.seed(42)
feature_maps = np.random.rand(8,8,16)
gradients    = np.random.randn(8,8,16)
alpha        = gradients.mean(axis=(0,1))
cam          = np.sum(feature_maps*alpha,axis=2)
cam          = np.maximum(cam,0)
cam          = cam/cam.max() if cam.max()>0 else cam

fig,axes=plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(np.random.rand(8,8),cmap='gray')
axes[0].set_title('Orijinal (sim.)'); axes[0].axis('off')
axes[1].imshow(cam,cmap='jet')
axes[1].set_title('Grad-CAM Haritası'); axes[1].axis('off')
# Overlay
axes[2].imshow(np.random.rand(8,8),cmap='gray',alpha=0.5)
axes[2].imshow(cam,cmap='jet',alpha=0.6)
axes[2].set_title('Üst üste (Overlay)'); axes[2].axis('off')
plt.suptitle('Grad-CAM — Model Neye Bakıyor?',fontweight='bold')
plt.tight_layout(); plt.show()
print(f"En yüksek aktivasyon: {np.unravel_index(cam.argmax(),cam.shape)}")


## 5. Alıştırmalar <a id='10'></a>

**1.** `conv2d` fonksiyonuna dilation (dilated convolution) desteği ekle: `dilation=2`.

**2.** 1×1 konvolüsyonun boyut azaltma etkisini göster: 64 kanal → 32 kanal.

**3.** VGG-16 mimariyi NumPy veya PyTorch ile yaz ve parametre sayısını hesapla.

**4.** ResNet bloğunu `projection shortcut` ile genişlet: giriş/çıkış boyutu farklıysa 1×1 konv ekle.

**5.** `conv2d` fonksiyonuna geri yayılım (backward) yaz ve gradyan kontrolü yap.

---
**Sonraki Modül:** Modül 07 — Transfer Learning
